# 07 · NLP de quejas (bloques J y K)

**Dónde:** quejas del área de atención (`data/simulados/quejas.csv`, simulación declarada, sin datos personales).
**Por qué:** enrutar cada queja a la cuadrilla correcta sin leerla primero, y encontrar quejas parecidas ya atendidas.
**Cómo:** limpieza y tokenización propias → TF-IDF (uni y bigramas) → regresión logística en un `Pipeline` de sklearn; similitud coseno para quejas parecidas.
**Con qué datos:** 1,400 quejas, 9 categorías. **Resultado:** tablas de abajo; las mismas cifras se publican en `analitica.resultados` (módulos `nlp` y `embeddings`).

Todo el código vive en `pipeline/src/nlp/`; este notebook solo lo ejecuta y muestra. Semilla 42.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parents[1]))
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
from pipeline.src.nlp.clasificador import cargar_quejas, textos_unicos, entrenar_y_evaluar, prioridad_desde_texto
from pipeline.src.nlp.similitud import comparar
from pipeline.src.nlp.texto import normalizar, tokenizar
df = cargar_quejas()
unicos = textos_unicos(df)
print(f"{len(df)} quejas · {len(unicos)} textos distintos · {df['categoria'].nunique()} categorías")
df['categoria'].value_counts().to_frame('quejas').join(unicos['categoria'].value_counts().rename('textos_distintos'))

1400 quejas · 331 textos distintos · 9 categorías


,quejas,textos_distintos
categoria,,
sin_agua,243,48
fuga_toma,240,23
fuga_calle,206,118
facturacion,155,12
baja_presion,145,40
drenaje,122,40
calidad_agua,122,21
medidor,92,11
atencion,75,18


## 1. Limpieza y tokenización
Minúsculas, sin acentos (se conserva la ñ), sin puntuación y sin palabras vacías. Se **conservan negaciones** (*no*, *sin*) porque cambian el sentido: *sin agua* ≠ *con agua*.

In [2]:
ejemplo = df['descripcion'].iloc[0]
print(ejemplo); print(normalizar(ejemplo)); print(tokenizar(ejemplo))

Desde hace dos semanas el chorro sale débil, sobre todo en la mañana.
desde hace dos semanas el chorro sale debil sobre todo en la mañana
['hace', 'semanas', 'chorro', 'sale', 'debil', 'mañana']


## 2. El riesgo principal: textos repetidos
Cada texto tiene una sola categoría y se repite en promedio ~4 veces. Si se divide por fila, el modelo ve en prueba textos idénticos a los de entrenamiento. Por eso **se divide por texto único** (regla (b) de la propuesta #5).

In [3]:
r = entrenar_y_evaluar()
print("Mejores hiperparámetros:", r.mejores_parametros)
print(f"F1 macro en validación cruzada (entrenamiento): {r.cv_f1_macro_media} ± {r.cv_f1_macro_desv}")
pd.DataFrame(r.metricas)

Mejores hiperparámetros: {'clf__C': 3.0, 'tfidf__ngram_range': [1, 2]}
F1 macro en validación cruzada (entrenamiento): 0.9782 ± 0.0306


,modelo,evaluacion,n_prueba,accuracy,f1_macro,textos_de_prueba_vistos_en_entrenamiento
0,TF-IDF + regresión logística,prueba por texto único,83,0.9639,0.9259,NaN
1,TF-IDF + regresión logística,"prueba por texto único, ponderada por quejas",363,0.8375,0.8382,NaN
2,Línea base (clase más frecuente),prueba por texto único,83,0.3614,0.0590,NaN
3,TF-IDF + regresión logística,"división ingenua por fila (CON FUGA, no válida)",350,1.0000,1.0000,0.9


## 3. Detalle por categoría y matriz de confusión (prueba por texto único)

In [4]:
r.por_categoria

,categoria,precision,recall,f1,textos_prueba
0,atencion,1.0,1.0,1.0000,4
1,baja_presion,1.0,1.0,1.0000,10
2,calidad_agua,1.0,1.0,1.0000,5
3,drenaje,1.0,1.0,1.0000,10
4,facturacion,1.0,1.0,1.0000,3
5,fuga_calle,1.0,1.0,1.0000,30
6,fuga_toma,1.0,0.5,0.6667,6
7,medidor,0.5,1.0,0.6667,3
8,sin_agua,1.0,1.0,1.0000,12


In [5]:
r.matriz

,atencion,baja_presion,calidad_agua,drenaje,facturacion,fuga_calle,fuga_toma,medidor,sin_agua
atencion,4,0,0,0,0,0,0,0,0
baja_presion,0,10,0,0,0,0,0,0,0
calidad_agua,0,0,5,0,0,0,0,0,0
drenaje,0,0,0,10,0,0,0,0,0
facturacion,0,0,0,0,3,0,0,0,0
fuga_calle,0,0,0,0,0,30,0,0,0
fuga_toma,0,0,0,0,0,0,3,3,0
medidor,0,0,0,0,0,0,0,3,0
sin_agua,0,0,0,0,0,0,0,0,12


## 4. Palabras clave que usa el modelo

In [6]:
r.palabras_clave

,categoria,terminos
0,atencion,"nadie, todavia, revisar, reporte hace, todavia no"
1,baja_presion,"presion, chorro, debil, chorro sale, sale debil"
2,calidad_agua,"agua sale, tierra, turbia, sale turbia, turbia..."
3,drenaje,"drenaje, desbordado huele, desbordado, muy mal..."
4,facturacion,"recibo, cobrando, habia pagado, cobrando perio..."
5,fuga_calle,"tirando toda, tirando, saliendo calle, agua sa..."
6,fuga_toma,"banqueta gotea, sin parar, gotea sin, gotea, l..."
7,medidor,"medidor, marca, llevaron medidor, toma no, caja"
8,sin_agua,"sin, abasto toda, toda calle, sin abasto, abasto"


## 5. ¿El texto determina la prioridad?
Insumo de diseño para el triage (T-12): si el texto no predice la prioridad, el triage no debe venderla como predicción.

In [7]:
pd.Series(prioridad_desde_texto())

textos_unicos                      331.0000
textos_con_mas_de_una_prioridad    155.0000
f1_macro_tfidf                       0.1791
f1_macro_tfidf_desv                  0.0387
f1_macro_azar_estratificado          0.2560
dtype: float64

## 6. Bloque K: quejas similares, TF-IDF frente a representaciones densas
precision@5 = fracción de los 5 vecinos más cercanos (en entrenamiento) que son de la misma categoría. Sentence-Transformers requiere descargar el modelo; si no hay acceso se marca *no ejecutado* y no se inventa cifra.

In [8]:
pd.DataFrame(comparar())

,representacion,dimension,ejecutado,precision_at_5,motivo
0,TF-IDF,166.0,True,0.894,NaN
1,"LSA (SVD de TF-IDF, 100 dims)",100.0,True,0.894,NaN
2,Sentence-Transformers (paraphrase-multilingual...,NaN,False,NaN,ModuleNotFoundError: No module named 'sentence...


## Conclusiones

In [9]:
p, pond, base, ing = r.metricas[:4]
prio = prioridad_desde_texto()
print(f"1. TF-IDF + regresión logística clasifica textos nunca vistos con F1 macro {p['f1_macro']:.3f} (línea base {base['f1_macro']:.3f}).")
from pipeline.src.nlp.resultados import _conclusion_matriz
print(f"2. Ponderado por quejas reales baja a {pond['f1_macro']:.3f} porque los textos que falla se repiten mucho. {_conclusion_matriz(r.matriz)}")
print(f"3. Dividir por fila daría {ing['f1_macro']:.3f}, cifra inflada por textos repetidos; por eso se divide por texto único.")
print(f"4. La prioridad no se deduce del texto (F1 {prio['f1_macro_tfidf']:.3f} vs azar {prio['f1_macro_azar_estratificado']:.3f}): el triage debe tratarla aparte.")

1. TF-IDF + regresión logística clasifica textos nunca vistos con F1 macro 0.926 (línea base 0.059).
2. Ponderado por quejas reales baja a 0.838 porque los textos que falla se repiten mucho. Confusiones: 3 de fuga_toma → medidor.
3. Dividir por fila daría 1.000, cifra inflada por textos repetidos; por eso se divide por texto único.
4. La prioridad no se deduce del texto (F1 0.179 vs azar 0.256): el triage debe tratarla aparte.
